In [47]:
from ribs.archives import ProximityArchive
from ribs.emitters import EmitterBase
from ribs.schedulers import Scheduler
from contextlib import contextmanager
import numpy as np

seed = 123
rng = np.random.default_rng(seed)

NOVELTY_THRESHOLD = 0.8

archive = ProximityArchive(
        solution_dim=1,
        measure_dim=2,
        k_neighbors=15,
        novelty_threshold=NOVELTY_THRESHOLD,
        seed=seed,
        local_competition=True
    )

In [48]:
class CustomEmitter(EmitterBase):
    def __init__(self, archive, batch_size=10, **kwargs):
        super().__init__(archive, solution_dim=1, bounds=None)
        self.batch_size = batch_size
        self.sol_count = 0

    def ask(self):
        solutions = []
        for _ in range(self.batch_size):
            solutions.append([self.sol_count])
            self.sol_count += 1
        return np.asarray(solutions)

In [49]:
@contextmanager
def _track_add_status(archive):
    substitutions = []   # list of (old_solution, new_solution)
    new_insertions = []  # list of new_solution
    original_store_add = archive._store.add

    def tracked_store_add(indices, data):
        indices_to_add = indices
        data_to_add = data
        
        occupied, old_data = archive._store.retrieve(indices)
        is_replacing = occupied
        is_new       = ~occupied
        
        # recheck to see if indices need to be replaced 
        replacing_indices = indices[is_replacing]
        new_indices = indices[is_new]
        
        replacing_data_objectives = data["objective"][is_replacing]
        old_objectives = old_data["objective"][is_replacing]
        
        if np.any(replacing_data_objectives <= old_objectives):
            details = {
                "indices": replacing_indices[replacing_data_objectives <= old_objectives],
                "new_objectives": replacing_data_objectives[replacing_data_objectives <= old_objectives],
                "old_objectives": old_objectives[replacing_data_objectives <= old_objectives],
            }
            print("Unexpectedly lower objective in add() call", details)
            
            is_replacing = False
            indices_to_add = new_indices  # only add the new indices, skip the replacements
            data_to_add = {k: v[is_new] for k, v in data.items()}
            
        if np.any(is_replacing):
            old_solutions = old_data["solution"][is_replacing].copy()
            new_solutions = data["solution"][is_replacing]
            substitutions.extend(zip(old_solutions, new_solutions))

        if np.any(is_new):
            new_insertions.extend(data["solution"][is_new])

        if len(indices_to_add) > 0:
            original_store_add(indices_to_add, data_to_add)
    
    archive._store.add = tracked_store_add
    try:
        yield new_insertions, substitutions
    finally:
        if "add" in archive._store.__dict__:
            del archive._store.__dict__["add"]

In [50]:
emitter = CustomEmitter(
        archive,
        batch_size=10,
        )
scheduler = Scheduler(archive, [emitter])


for _ in range(1000):
    sols = scheduler.ask()
    measures_batch = []
    objectives_batch = []
    for sol in sols:
        measure = rng.uniform(low=0.0, high=10.0, size=2)
        objective = rng.normal(loc=10.0, scale=5.0)
        measures_batch.append(measure)
        objectives_batch.append(objective)

    with _track_add_status(archive) as (new_insertions, substitutions):
        scheduler.tell(list(objectives_batch), list(measures_batch))
        
    #print counts 
    print(f"New insertions: {len(new_insertions)}, Substitutions: {len(substitutions)}, Current archive size: {len(archive)}")

New insertions: 10, Substitutions: 0, Current archive size: 10
New insertions: 10, Substitutions: 0, Current archive size: 20
New insertions: 10, Substitutions: 0, Current archive size: 30
New insertions: 10, Substitutions: 0, Current archive size: 40
New insertions: 10, Substitutions: 0, Current archive size: 50
New insertions: 10, Substitutions: 0, Current archive size: 60
New insertions: 10, Substitutions: 0, Current archive size: 70
New insertions: 10, Substitutions: 0, Current archive size: 80
New insertions: 10, Substitutions: 0, Current archive size: 90
New insertions: 10, Substitutions: 0, Current archive size: 100
New insertions: 10, Substitutions: 0, Current archive size: 110
New insertions: 10, Substitutions: 0, Current archive size: 120
New insertions: 10, Substitutions: 0, Current archive size: 130
New insertions: 10, Substitutions: 0, Current archive size: 140
New insertions: 10, Substitutions: 0, Current archive size: 150
New insertions: 10, Substitutions: 0, Current arc

#### We repopulate a new archive with the "elites" of the old archive using two different strategies
##### 1. Use standard add() calls with the same parameters
##### 2. Calculate the KNN and novelty values ourselves keeping in consideration the whole "elite" population and then call add() in a single batch.

In [51]:
NEW_BATCH_SIZE = 10

new_archive = ProximityArchive(
        solution_dim=1,
        measure_dim=2,
        k_neighbors=15,
        novelty_threshold=NOVELTY_THRESHOLD,
        seed=seed,
        local_competition=True
    )

emitter = CustomEmitter(
        new_archive,
        batch_size=NEW_BATCH_SIZE,
        )
scheduler = Scheduler(new_archive, [emitter])

solutions = archive.data("solution")
objectives = archive.data("objective")
measures = archive.data("measures")

# shuffle the solutions
shuffle = True
if shuffle:
    indices = np.arange(len(solutions))
    # shuffle with seed 1233
    rng.shuffle(indices)
    solutions = solutions[indices]
    measures = measures[indices]
    objectives = objectives[indices]

for i in range(len(solutions) // NEW_BATCH_SIZE + 1):
    sols = solutions[i*NEW_BATCH_SIZE:(i+1)*NEW_BATCH_SIZE]
    measures_batch = measures[i*NEW_BATCH_SIZE:(i+1)*NEW_BATCH_SIZE]
    objectives_batch = objectives[i*NEW_BATCH_SIZE:(i+1)*NEW_BATCH_SIZE]
    if len(sols) < NEW_BATCH_SIZE:
        continue
    _ = scheduler.ask()
    
    with _track_add_status(new_archive) as (new_insertions, substitutions):
        scheduler.tell(list(objectives_batch), list(measures_batch))
        
    #print counts 
    print(f"New insertions: {len(new_insertions)}, Substitutions: {len(substitutions)}, Current archive size: {len(new_archive)}")

New insertions: 10, Substitutions: 0, Current archive size: 10
New insertions: 10, Substitutions: 0, Current archive size: 20
New insertions: 10, Substitutions: 0, Current archive size: 30
New insertions: 10, Substitutions: 0, Current archive size: 40
New insertions: 10, Substitutions: 0, Current archive size: 50
New insertions: 10, Substitutions: 0, Current archive size: 60
New insertions: 10, Substitutions: 0, Current archive size: 70
New insertions: 10, Substitutions: 0, Current archive size: 80
New insertions: 10, Substitutions: 0, Current archive size: 90
New insertions: 10, Substitutions: 0, Current archive size: 100
New insertions: 10, Substitutions: 0, Current archive size: 110
New insertions: 10, Substitutions: 0, Current archive size: 120
New insertions: 10, Substitutions: 0, Current archive size: 130
New insertions: 10, Substitutions: 0, Current archive size: 140
New insertions: 10, Substitutions: 0, Current archive size: 150
New insertions: 10, Substitutions: 0, Current arc

In [52]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

solutions = archive.data("solution")
objectives = archive.data("objective")
measures = archive.data("measures")

k = 15
nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
nn.fit(measures)

distances, indices = nn.kneighbors(measures)

# exclude self (index 0)
novelty_scores = distances[:, 1:].mean(axis=1)
nearest_neighbor_idx = indices[:, 1]  # closest other solution

novel_mask     = novelty_scores > NOVELTY_THRESHOLD
non_novel_mask = ~novel_mask

sol_to_add     = []
obj_to_add     = []
measure_to_add = []

# --- Pass 1: novel solutions, always admitted ---
for i in np.where(novel_mask)[0]:
    sol_to_add.append(solutions[i])
    obj_to_add.append(objectives[i])
    measure_to_add.append(measures[i])

# --- Pass 2: non-novel solutions, simulate local competition ---
# Group colliding solutions by which neighbor they're competing against
collision_groups = {}   # nn_idx -> [(objective, solution_idx), ...]
for i in np.where(non_novel_mask)[0]:
    nn_idx = nearest_neighbor_idx[i]
    collision_groups.setdefault(nn_idx, []).append((objectives[i], i))

n_collision_winners = 0
for nn_idx, competitors in collision_groups.items():
    # best challenger in this collision group
    best_obj, best_i = max(competitors, key=lambda x: x[0])
    
    # wins only if it beats the incumbent's objective
    if best_obj > objectives[nn_idx]:
        sol_to_add.append(solutions[best_i])
        obj_to_add.append(objectives[best_i])
        measure_to_add.append(measures[best_i])
        n_collision_winners += 1

print(f"Novel solutions added   : {novel_mask.sum()}")
print(f"Collision winners added : {n_collision_winners}")
print(f"Total solutions to add  : {len(sol_to_add)}")

Novel solutions added   : 65
Collision winners added : 209
Total solutions to add  : 274
